# 05 Measure Relationships

Assignment step 8 bonus: generate grounded, typed candidate relationships between final measures. `narrower_than` is represented as the inverse reading of exported `broader_than` edges.


In [1]:
import csv
import json
from pathlib import Path

import pyarrow.parquet as pq

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent


def read_json(relative_path: str):
    path = ROOT / relative_path
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {"missing": str(path)}


def csv_rows(relative_path: str, limit: int | None = None):
    csv.field_size_limit(2_147_483_647)
    path = ROOT / relative_path
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8-sig", newline="") as file:
        rows = list(csv.DictReader(file))
    return rows if limit is None else rows[:limit]


def csv_count(relative_path: str) -> int | None:
    path = ROOT / relative_path
    if not path.exists():
        return None
    return len(csv_rows(relative_path))


def parquet_count(relative_path: str) -> int | None:
    path = ROOT / relative_path
    return pq.read_table(path).num_rows if path.exists() else None

## Relationship Summary


In [2]:
metrics = read_json("report/relations_metrics.json")
{
    "relation_count": metrics.get("relation_count"),
    "relation_type_distribution": metrics.get("relation_type_distribution"),
    "validation": metrics.get("validation"),
    "manual_review": metrics.get("manual_review"),
    "latest_run_summary": metrics.get("latest_run_summary"),
}

{'relation_count': 30695,
 'relation_type_distribution': {'broader_than': 16382,
  'related_to': 13482,
  'variant_of': 831},
 'validation': {'failures': [], 'passed': True},
 'manual_review': {'completed_count': 100,
  'false_positive_taxonomy': {'denominator_fragment': 2,
   'denominator_mismatch': 1,
   'generic_qualifier': 4,
   'lexical_role_conflict': 1,
   'overbroad_source': 1,
   'semantic_drift': 2},
  'precision_at_10': 1.0,
  'precision_at_100': 0.89,
  'precision_at_25': 1.0,
  'precision_at_50': 0.96,
  'review_sample': 'outputs\\relations\\run_e6e523656ea9857ecac9\\manual_relation_review_sample.csv',
  'sample_count': 100,
  'status': 'available',
  'typed_accuracy': 0.8876404494382022},
 'latest_run_summary': {'accepted_count': 30695,
  'all_pair_count': 4186171,
  'candidate_count': 45351,
  'candidate_generation_seconds': 1.2998695999995107,
  'candidate_pair_generation_seconds': 25.50670539999919,
  'confidence_bands': {'0.00-0.69': 114,
   '0.70-0.84': 26832,
   '0.

## Relationship Examples


In [3]:
keep = ["relation_id", "source_term", "target_term", "relation_type", "confidence"]
[
    {key: row.get(key) for key in keep if key in row}
    for row in csv_rows("outputs/measure_relations.csv", 8)
]

[{'relation_id': 'relation_04627d6eb77a6477146f',
  'source_term': 'Long-term unemployment rate by sex',
  'target_term': 'Long-term unemployment rates by sex',
  'relation_type': 'related_to',
  'confidence': '0.961892'},
 {'relation_id': 'relation_96fb5d93072be928ca26',
  'source_term': 'Long-term unemployment rate by',
  'target_term': 'Long-term unemployment rates by',
  'relation_type': 'related_to',
  'confidence': '0.958677'},
 {'relation_id': 'relation_98f91175ed65e9bd3a7f',
  'source_term': 'Long-term unemployment rate',
  'target_term': 'long-term unemployment rates',
  'relation_type': 'related_to',
  'confidence': '0.956055'},
 {'relation_id': 'relation_d0623cf6623e0565320f',
  'source_term': 'Unemployment rate by',
  'target_term': 'Unemployment rates by',
  'relation_type': 'related_to',
  'confidence': '0.954751'},
 {'relation_id': 'relation_8b9ba6e70d7fa6464344',
  'source_term': 'Gross value added - NACE Rev. 2: C',
  'target_term': 'Gross value added - NACE Rev. 2: F'